# 05 - Player-Input Rolls + Agency-First DM Loop

This notebook combines patterns from 01-04 and adds DM-style roll timing:
- **forced checks** for involuntary danger (for example poison exposure)
- **scene checks** sometimes triggered by entering a new area
- **intent checks** only after the player states an approach (for example locked box: pick vs smash)

Interactive tools:
- `prompt_player_action`: asks the player what they do (open-ended)
- `roll_dice`: asks the player for raw d20 input and returns structured mechanics

Reading order:
1. Setup
2. Character/world/run state
3. Tool implementations
4. Tool schemas + policies
5. Trace helpers + dispatcher
6. Agency + roll-timing system prompt
7. Main loop
8. Demo run


## Step 1 - Setup

Configure adapter access, model constants, and verbose tracing toggles.


In [1]:
# Section: Setup
# Purpose: Import dependencies, configure tracing, and initialize the adapter.

import json
import re
import sys
import time
from pathlib import Path
from typing import Any

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "orchestrator").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from orchestrator.llm_interaction.adapter import LLMAdapter, LLMError

MODEL = "gpt-oss:20b"
MAX_ITERATIONS = 80
TRACE_PREVIEW_CHARS = 700

FULL_TRACE = True
SHOW_THINKING_TRACE = True
SHOW_RAW_RESPONSE = True
SHOW_RESPONSE_STATS = True
PRINT_FULL_TRACE_AFTER_RUN = True
PRINT_FULL_RESPONSE_SNAPSHOTS_AFTER_RUN = False

adapter = LLMAdapter(model=MODEL, verbose=False)


## Step 2 - Character, World, and Runtime State

Defines source-of-truth state and mutable run counters.


In [2]:
# Section: Runtime state
# Purpose: Define character/world data and mutable counters for policy and reporting.

CHARACTER_SHEET = {
    "name": "Nyx Embervale",
    "class": "Rogue",
    "level": 5,
    "stats": {
        "strength": 10,
        "dexterity": 17,
        "constitution": 13,
        "intelligence": 14,
        "wisdom": 12,
        "charisma": 15,
    },
    "skills": {
        "acrobatics": "dexterity",
        "athletics": "strength",
        "deception": "charisma",
        "insight": "wisdom",
        "investigation": "intelligence",
        "perception": "wisdom",
        "persuasion": "charisma",
        "sleight_of_hand": "dexterity",
        "stealth": "dexterity",
        "survival": "wisdom",
    },
}

WORLD_STATE = {
    "location": "The Shattered Beacon",
    "scene_goal": "Recover the storm ledger from the archive vault.",
    "tension": "high",
    "time_of_day": "midnight",
    "elements": [
        "A locked iron strongbox on a stone desk (optional lead)",
        "A tray with a suspicious wine carafe",
        "Faint scratch marks near a hidden wall panel",
        "Open storm maps pinned across a table",
        "Distant footsteps in the stairwell",
    ],
}

RUN_STATE = {
    "total_tool_calls": 0,
    "check_log": [],
    "session_notes": [],
    "last_observed": {},
    "rolls_requested": 0,
    "actions_prompted": 0,
}

ENCOUNTER_PROMPT = (
    "Begin the Shattered Beacon scene with improvisational pacing. "
    "Keep dice optional and only call for rolls when uncertainty and stakes justify it. "
    "Do not steer me toward any single obstacle; follow my declared intent and let me ignore or revisit hooks freely."
)


def reset_run_state() -> None:
    RUN_STATE["total_tool_calls"] = 0
    RUN_STATE["check_log"] = []
    RUN_STATE["session_notes"] = []
    RUN_STATE["last_observed"] = {}
    RUN_STATE["rolls_requested"] = 0
    RUN_STATE["actions_prompted"] = 0


def ability_modifier(score: int) -> int:
    return (int(score) - 10) // 2


def world_summary() -> str:
    return json.dumps(WORLD_STATE, indent=2)


## Step 3 - Tool Implementations

Key interaction tools:
- `prompt_player_action` asks for free-form player intent.
- `roll_dice` asks for the d20 face value and records check context.


In [3]:
# Section: Tool implementations
# Purpose: Provide state tools, intent-capture tools, and player-input roll mechanics.

def read_character_sheet() -> dict[str, Any]:
    return json.loads(json.dumps(CHARACTER_SHEET))


def read_world_state() -> dict[str, Any]:
    return json.loads(json.dumps(WORLD_STATE))


def set_world_state(
    location: str | None = None,
    scene_goal: str | None = None,
    tension: str | None = None,
    time_of_day: str | None = None,
) -> dict[str, Any]:
    if location:
        WORLD_STATE["location"] = location
    if scene_goal:
        WORLD_STATE["scene_goal"] = scene_goal
    if tension:
        WORLD_STATE["tension"] = tension
    if time_of_day:
        WORLD_STATE["time_of_day"] = time_of_day
    return json.loads(json.dumps(WORLD_STATE))


def get_skill_stat(skill: str) -> dict[str, Any]:
    key = str(skill).strip().lower()
    skills = CHARACTER_SHEET["skills"]
    if key not in skills:
        return {
            "ok": False,
            "error": f"Unknown skill '{skill}'. Available: {', '.join(sorted(skills.keys()))}",
        }
    return {"ok": True, "skill": key, "stat": skills[key]}


def get_stat_modifier(stat: str) -> dict[str, Any]:
    key = str(stat).strip().lower()
    stats = CHARACTER_SHEET["stats"]
    if key not in stats:
        return {
            "ok": False,
            "error": f"Unknown stat '{stat}'. Available: {', '.join(sorted(stats.keys()))}",
        }
    score = int(stats[key])
    return {
        "ok": True,
        "stat": key,
        "score": score,
        "modifier": ability_modifier(score),
    }


def prompt_player_action(prompt: str = "What do you do?") -> dict[str, Any]:
    """Interactive tool: asks the player for an open-ended action declaration."""
    q = str(prompt).strip() or "What do you do?"

    RUN_STATE["actions_prompted"] += 1

    print("\n" + "-" * 72, flush=True)
    print("DM QUESTION", flush=True)
    print(q, flush=True)

    while True:
        action = input("Player action: ").strip()
        if action.lower() in {"q", "quit", "exit"}:
            raise RuntimeError("Player aborted during action prompt.")
        if action:
            break
        print("Please enter an action or 'q' to abort.", flush=True)

    RUN_STATE["last_observed"]["player_intent"] = action

    print(f"Captured action: {action}", flush=True)
    print("-" * 72 + "\n", flush=True)

    return {
        "player_action": action,
        "prompt": q,
    }


def roll_dice(
    skill: str,
    dc: int,
    modifier: int,
    check_type: str,
    dm_prompt: str = "",
    player_intent: str = "",
    consequence_on_fail: str = "",
) -> dict[str, Any]:
    """Interactive tool: asks player for d20 value after DM calls for a check."""
    skill_key = str(skill).strip().lower()
    target_dc = int(dc)
    mod = int(modifier)
    ctype = str(check_type).strip().lower()
    intent = str(player_intent).strip()
    fail_consequence = str(consequence_on_fail).strip()

    if dm_prompt.strip():
        prompt_text = dm_prompt.strip()
    elif ctype == "forced_check":
        prompt_text = f"This is a forced {skill_key} check. Roll now."
    elif ctype == "scene_check":
        prompt_text = f"Before you proceed, give me a quick {skill_key} check."
    else:
        if intent:
            prompt_text = f"Since you attempt '{intent}', make a {skill_key} check."
        else:
            prompt_text = f"Make a {skill_key} check."

    RUN_STATE["rolls_requested"] += 1

    print("\n" + "=" * 72, flush=True)
    print("DM CALLS FOR A ROLL", flush=True)
    print(prompt_text, flush=True)
    print(f"Check type  : {ctype}", flush=True)
    print(f"Skill       : {skill_key}", flush=True)
    print(f"DC          : {target_dc}", flush=True)
    print(f"Modifier    : {mod:+d}", flush=True)
    if intent:
        print(f"Player intent: {intent}", flush=True)
    if fail_consequence:
        print(f"Fail state  : {fail_consequence}", flush=True)
    print("Enter raw d20 value (1-20), or 'q' to abort.", flush=True)

    while True:
        raw = input(f"d20 roll for {skill_key}: ").strip()

        if raw.lower() in {"q", "quit", "exit"}:
            raise RuntimeError("Player aborted manual roll input.")

        try:
            roll_value = int(raw)
        except ValueError:
            print("Invalid input. Enter an integer from 1 to 20.", flush=True)
            continue

        if not 1 <= roll_value <= 20:
            print("Out of range. Enter a value from 1 to 20.", flush=True)
            continue

        break

    total = roll_value + mod
    success = total >= target_dc

    result = {
        "check_type": ctype,
        "skill": skill_key,
        "dc": target_dc,
        "modifier": mod,
        "player_roll": roll_value,
        "total": total,
        "success": success,
        "player_intent": intent,
        "dm_prompt": prompt_text,
        "consequence_on_fail": fail_consequence,
    }

    RUN_STATE["last_observed"].update(
        {
            "check_type": ctype,
            "skill": skill_key,
            "dc": target_dc,
            "modifier": mod,
            "roll": roll_value,
            "total": total,
            "success": success,
            "player_intent": intent,
            "dm_prompt": prompt_text,
            "consequence_on_fail": fail_consequence,
        }
    )

    print(f"Recorded roll: d20={roll_value}, total={total}, success={success}", flush=True)
    print("=" * 72 + "\n", flush=True)

    return result


def record_skill_check(
    skill: str,
    stat: str,
    dc: int,
    roll: int,
    modifier: int,
    total: int,
    success: bool,
    check_type: str = "intent_check",
    player_intent: str = "",
    dm_prompt: str = "",
    consequence_on_fail: str = "",
    note: str = "",
) -> dict[str, Any]:
    entry = {
        "index": len(RUN_STATE["check_log"]) + 1,
        "check_type": str(check_type).strip().lower(),
        "skill": str(skill).strip().lower(),
        "stat": str(stat).strip().lower(),
        "dc": int(dc),
        "roll": int(roll),
        "modifier": int(modifier),
        "total": int(total),
        "success": bool(success),
        "player_intent": str(player_intent).strip(),
        "dm_prompt": str(dm_prompt).strip(),
        "consequence_on_fail": str(consequence_on_fail).strip(),
        "note": str(note).strip(),
    }
    RUN_STATE["check_log"].append(entry)
    if entry["note"]:
        RUN_STATE["session_notes"].append(entry["note"])
    return {
        "recorded": entry,
        "completed_checks": len(RUN_STATE["check_log"]),
    }


def add_session_note(text: str) -> dict[str, Any]:
    note = str(text).strip()
    if not note:
        raise ValueError("text cannot be empty")
    RUN_STATE["session_notes"].append(note)
    return {
        "ok": True,
        "note": note,
        "total_notes": len(RUN_STATE["session_notes"]),
    }


def get_progress() -> dict[str, Any]:
    checks = RUN_STATE["check_log"]
    latest = checks[-1] if checks else None
    return {
        "completed_checks": len(checks),
        "total_tool_calls": RUN_STATE["total_tool_calls"],
        "rolls_requested": RUN_STATE["rolls_requested"],
        "actions_prompted": RUN_STATE["actions_prompted"],
        "latest_check": latest,
    }


## Step 4 - Tool Schemas and Runtime Policies

- `pre_tool_use` validates roll timing contracts.
- `stop_policy` blocks menu-style output and enforces open-ended agency.


In [4]:
# Section: Schemas + policies
# Purpose: Expose tool contracts and enforce roll-timing + agency guardrails.

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "read_character_sheet",
            "description": "Read the full player character sheet.",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_world_state",
            "description": "Read current world state.",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "set_world_state",
            "description": "Update world state fields.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {"type": "string"},
                    "scene_goal": {"type": "string"},
                    "tension": {"type": "string"},
                    "time_of_day": {"type": "string"},
                },
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "prompt_player_action",
            "description": "Prompt the player for an open-ended action declaration.",
            "parameters": {
                "type": "object",
                "properties": {"prompt": {"type": "string"}},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_skill_stat",
            "description": "Resolve which stat applies to a skill.",
            "parameters": {
                "type": "object",
                "properties": {"skill": {"type": "string"}},
                "required": ["skill"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_stat_modifier",
            "description": "Resolve stat score and modifier.",
            "parameters": {
                "type": "object",
                "properties": {"stat": {"type": "string"}},
                "required": ["stat"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "roll_dice",
            "description": "Ask player for d20 input after DM calls for a check.",
            "parameters": {
                "type": "object",
                "properties": {
                    "skill": {"type": "string"},
                    "dc": {"type": "integer", "minimum": 5, "maximum": 30},
                    "modifier": {"type": "integer", "minimum": -10, "maximum": 15},
                    "check_type": {
                        "type": "string",
                        "enum": ["forced_check", "scene_check", "intent_check"],
                    },
                    "dm_prompt": {"type": "string"},
                    "player_intent": {"type": "string"},
                    "consequence_on_fail": {"type": "string"},
                },
                "required": ["skill", "dc", "modifier", "check_type"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "record_skill_check",
            "description": "Record one resolved check.",
            "parameters": {
                "type": "object",
                "properties": {
                    "skill": {"type": "string"},
                    "stat": {"type": "string"},
                    "dc": {"type": "integer", "minimum": 5, "maximum": 30},
                    "roll": {"type": "integer", "minimum": 1, "maximum": 20},
                    "modifier": {"type": "integer", "minimum": -10, "maximum": 15},
                    "total": {"type": "integer", "minimum": -10, "maximum": 40},
                    "success": {"type": "boolean"},
                    "check_type": {"type": "string"},
                    "player_intent": {"type": "string"},
                    "dm_prompt": {"type": "string"},
                    "consequence_on_fail": {"type": "string"},
                    "note": {"type": "string"},
                },
                "required": ["skill", "stat", "dc", "roll", "modifier", "total", "success"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "add_session_note",
            "description": "Add a short persistent note.",
            "parameters": {
                "type": "object",
                "properties": {"text": {"type": "string"}},
                "required": ["text"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_progress",
            "description": "Read loop progress and counters.",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
]


def pre_tool_use(tool_name: str, args: dict[str, Any]) -> dict[str, Any]:
    if tool_name == "roll_dice":
        dc = int(args.get("dc", 0))
        if dc < 5 or dc > 30:
            return {"allow": False, "reason": "DC must be between 5 and 30."}

        check_type = str(args.get("check_type", "")).strip().lower()
        allowed_types = {"forced_check", "scene_check", "intent_check"}
        if check_type not in allowed_types:
            return {
                "allow": False,
                "reason": f"check_type must be one of {sorted(allowed_types)}.",
            }

        intent = str(args.get("player_intent", "")).strip()
        if check_type == "intent_check" and not intent:
            return {
                "allow": False,
                "reason": "intent_check requires player_intent so rolls happen after declared approach.",
            }

        if check_type == "scene_check" and RUN_STATE.get("rolls_requested", 0) >= 1:
            return {
                "allow": False,
                "reason": "Avoid repeated unsolicited scene checks. Let the player act before another scene_check.",
            }

        unresolved_roll_pressure = RUN_STATE.get("rolls_requested", 0) - len(RUN_STATE.get("check_log", []))
        if unresolved_roll_pressure > 2:
            return {
                "allow": False,
                "reason": "Too many roll requests in sequence. Narrate and re-anchor player agency before another roll.",
            }

    if tool_name == "record_skill_check" and "success" not in args:
        return {"allow": False, "reason": "record_skill_check requires explicit success boolean."}

    return {"allow": True}


def post_tool_use(tool_name: str, payload: dict[str, Any]) -> str | None:
    if payload.get("ok", True):
        return None

    if tool_name == "roll_dice":
        return "Roll input failed. Re-issue roll_dice with valid arguments and wait for player input."

    return "A tool failed. Correct arguments and continue."


def stop_policy(assistant_text: str) -> str | None:
    text = (assistant_text or "").strip()
    lowered = text.lower()

    # Ban menu-style agency narrowing.
    markers = [r"(?im)^\s*\d+[\.)]\s+", r"(?im)^\s*[-*]\s+"]
    for marker in markers:
        if re.search(marker, text):
            return "Do not provide menu-style options or bullet lists."

    banned_terms = ["choose one", "options:", "you can:", "pick one"]
    if any(term in lowered for term in banned_terms):
        return "Do not enumerate player choices. Keep agency open-ended."

    open_prompt_phrases = ["what do you do", "how do you proceed", "what is your move", "what's your move", "your move"]
    has_open_prompt = any(p in lowered for p in open_prompt_phrases) or "?" in text
    if not has_open_prompt:
        return "End with an open-ended invitation for player action."

    return None


## Step 5 - Trace Helpers and Tool Dispatcher

Normalizes model responses, logs every step, executes tools, and keeps observed mechanics context.


In [5]:
# Section: Tracing + dispatch
# Purpose: Provide verbose logging and deterministic tool execution glue.

TOOL_IMPL = {
    "read_character_sheet": read_character_sheet,
    "read_world_state": read_world_state,
    "set_world_state": set_world_state,
    "prompt_player_action": prompt_player_action,
    "get_skill_stat": get_skill_stat,
    "get_stat_modifier": get_stat_modifier,
    "roll_dice": roll_dice,
    "record_skill_check": record_skill_check,
    "add_session_note": add_session_note,
    "get_progress": get_progress,
}


def ts() -> str:
    return time.strftime("%H:%M:%S")


def preview(text: str, limit: int = TRACE_PREVIEW_CHARS) -> str:
    if text is None:
        return ""
    s = str(text).strip()
    if FULL_TRACE or len(s) <= limit:
        return s
    return s[:limit] + f" ... [truncated {len(s) - limit} chars]"


def trace_print(tag: str, message: str, trace_lines: list[str] | None = None) -> None:
    line = f"[{ts()}] [{tag}] {message}"
    print(line, flush=True)
    if trace_lines is not None:
        trace_lines.append(line)


def response_to_dict(response: Any) -> dict[str, Any]:
    if hasattr(response, "model_dump"):
        try:
            data = response.model_dump(exclude_none=False)
            if isinstance(data, dict):
                return data
        except Exception:
            pass

    if isinstance(response, dict):
        return response

    return {
        "_type": type(response).__name__,
        "repr": repr(response),
    }


def extract_message_dict(response: Any) -> dict[str, Any]:
    data = response_to_dict(response)
    msg = data.get("message") if isinstance(data, dict) else None
    if isinstance(msg, dict):
        return msg
    if hasattr(msg, "model_dump"):
        try:
            dumped = msg.model_dump(exclude_none=False)
            if isinstance(dumped, dict):
                return dumped
        except Exception:
            pass
    return {}


def extract_thinking_text(response: Any) -> str:
    msg = extract_message_dict(response)
    thinking = msg.get("thinking", "")
    if isinstance(thinking, list):
        return "".join(str(x) for x in thinking)
    return str(thinking or "")


def normalize_tool_calls(response: Any) -> list[dict[str, Any]]:
    out: list[dict[str, Any]] = []
    raw_calls = adapter._extract_tool_calls(response)

    for i, call in enumerate(raw_calls):
        if not isinstance(call, dict):
            continue

        fn = call.get("function", {})
        name = fn.get("name") or call.get("name")
        args = fn.get("arguments", {})

        if isinstance(args, str):
            try:
                args = json.loads(args)
            except json.JSONDecodeError:
                args = {}

        if not isinstance(args, dict) or not name:
            continue

        out.append(
            {
                "id": call.get("id") or f"call_{i}",
                "name": str(name),
                "arguments": args,
                "raw": call,
                "source": "structured",
            }
        )

    return out


def extract_fallback_tool_calls_from_text(text: str) -> list[dict[str, Any]]:
    if not text:
        return []

    calls: list[dict[str, Any]] = []

    for line in text.splitlines():
        stripped = line.strip().rstrip(",")
        if not stripped.startswith("{") or not stripped.endswith("}"):
            continue

        try:
            obj = json.loads(stripped)
        except json.JSONDecodeError:
            continue

        if not isinstance(obj, dict):
            continue

        name = obj.get("name")
        params = obj.get("parameters", {})
        if not name or not isinstance(params, dict):
            continue

        calls.append(
            {
                "id": f"text_line_{len(calls)}",
                "name": str(name),
                "arguments": params,
                "raw": {"function": {"name": str(name), "arguments": params}},
                "source": "text_fallback",
            }
        )

    pattern = re.compile(
        r'\{\s*"name"\s*:\s*"(?P<name>[^"]+)"\s*,\s*"parameters"\s*:\s*(?P<params>\{.*?\})\s*\}',
        flags=re.DOTALL,
    )

    for match in pattern.finditer(text):
        name = match.group("name")
        params_raw = match.group("params")
        try:
            params = json.loads(params_raw)
        except json.JSONDecodeError:
            continue
        if not isinstance(params, dict):
            continue

        duplicate = any(c["name"] == str(name) and c["arguments"] == params for c in calls)
        if duplicate:
            continue

        calls.append(
            {
                "id": f"text_regex_{len(calls)}",
                "name": str(name),
                "arguments": params,
                "raw": {"function": {"name": str(name), "arguments": params}},
                "source": "text_fallback",
            }
        )

    return calls


def update_last_observed(name: str, payload: dict[str, Any]) -> None:
    if not payload.get("ok"):
        return

    result = payload.get("result")
    if not isinstance(result, dict):
        return

    observed = RUN_STATE.setdefault("last_observed", {})

    if name == "prompt_player_action":
        observed["player_intent"] = result.get("player_action", "")
    elif name == "get_skill_stat" and result.get("ok"):
        observed["skill"] = result.get("skill")
        observed["stat"] = result.get("stat")
    elif name == "get_stat_modifier" and result.get("ok"):
        observed["stat"] = result.get("stat")
        observed["modifier"] = result.get("modifier")
    elif name == "roll_dice":
        observed["check_type"] = result.get("check_type")
        observed["skill"] = result.get("skill")
        observed["dc"] = result.get("dc")
        observed["modifier"] = result.get("modifier")
        observed["roll"] = result.get("player_roll")
        observed["total"] = result.get("total")
        observed["success"] = result.get("success")
        observed["player_intent"] = result.get("player_intent")
        observed["dm_prompt"] = result.get("dm_prompt")
        observed["consequence_on_fail"] = result.get("consequence_on_fail")


def hydrate_record_skill_check_args(arguments: dict[str, Any]) -> dict[str, Any]:
    observed = RUN_STATE.get("last_observed", {})
    merged = dict(arguments)

    skill = str(merged.get("skill") or observed.get("skill") or "perception").strip().lower()
    stat = str(merged.get("stat") or observed.get("stat") or CHARACTER_SHEET["skills"].get(skill, "wisdom")).strip().lower()
    dc = int(merged.get("dc", observed.get("dc", 12)))
    modifier = int(merged.get("modifier", observed.get("modifier", ability_modifier(CHARACTER_SHEET["stats"].get(stat, 10)))))
    roll = int(merged.get("roll", observed.get("roll", 10)))
    total = int(merged.get("total", observed.get("total", roll + modifier)))
    success = bool(merged.get("success", observed.get("success", total >= dc)))

    check_type = str(merged.get("check_type") or observed.get("check_type") or "intent_check").strip().lower()
    player_intent = str(merged.get("player_intent") or observed.get("player_intent") or "").strip()
    dm_prompt = str(merged.get("dm_prompt") or observed.get("dm_prompt") or "").strip()
    consequence_on_fail = str(merged.get("consequence_on_fail") or observed.get("consequence_on_fail") or "").strip()
    note = str(merged.get("note", "")).strip()

    return {
        "skill": skill,
        "stat": stat,
        "dc": dc,
        "roll": roll,
        "modifier": modifier,
        "total": total,
        "success": success,
        "check_type": check_type,
        "player_intent": player_intent,
        "dm_prompt": dm_prompt,
        "consequence_on_fail": consequence_on_fail,
        "note": note,
    }


def execute_tool(name: str, arguments: dict[str, Any], trace_lines: list[str]) -> dict[str, Any]:
    RUN_STATE["total_tool_calls"] += 1
    call_no = RUN_STATE["total_tool_calls"]

    trace_print("TOOL-CALL", f"#{call_no} {name} args={preview(json.dumps(arguments, ensure_ascii=True))}", trace_lines)

    pre = pre_tool_use(name, arguments)
    trace_print("HOOK-PRE", f"{name} -> {json.dumps(pre, ensure_ascii=True)}", trace_lines)
    if not pre.get("allow", True):
        payload = {"ok": False, "error": pre.get("reason", "Blocked by pre_tool_use")}
        trace_print("TOOL-RESULT", f"#{call_no} {name} -> {preview(json.dumps(payload, ensure_ascii=True))}", trace_lines)
        return payload

    fn = TOOL_IMPL.get(name)
    if fn is None:
        payload = {"ok": False, "error": f"Unknown tool: {name}"}
        trace_print("TOOL-RESULT", f"#{call_no} {name} -> {payload['error']}", trace_lines)
        return payload

    call_args = dict(arguments)
    if name == "record_skill_check":
        call_args = hydrate_record_skill_check_args(call_args)
        trace_print("TOOL-NORMALIZE", f"record_skill_check hydrated args={preview(json.dumps(call_args, ensure_ascii=True))}", trace_lines)

    try:
        payload = {"ok": True, "result": fn(**call_args)}
    except Exception as exc:
        payload = {"ok": False, "error": str(exc)}

    update_last_observed(name, payload)

    post_note = post_tool_use(name, payload)
    if post_note:
        payload["hook_note"] = post_note
        trace_print("HOOK-POST", post_note, trace_lines)

    trace_print("TOOL-RESULT", f"#{call_no} {name} -> {preview(json.dumps(payload, ensure_ascii=True))}", trace_lines)
    return payload


## Step 6 - System Prompt (Agency + Roll Timing)

This prompt makes roll timing explicit:
- forced checks for involuntary hazards
- occasional scene checks on entry when stakes justify
- intent checks only after player states an approach


In [6]:
# Section: System prompt
# Purpose: Encode agency-first narration and low-pressure, situational roll timing.

SYSTEM_PROMPT = (
    "You are a Dungeon Master runtime focused on player agency, improvisation, and fair mechanics.\n\n"
    "Player agency rules:\n"
    "1. Never decide the player's choices, speech, tactics, or intent for them.\n"
    "2. Never provide numbered options, bullet menus, or choose-one lists.\n"
    "3. Narrate the world and consequences, then leave control with an open-ended prompt.\n"
    "4. Do not steer the player toward any single obstacle repeatedly. If they ignore a hook, move on naturally.\n\n"
    "Roll timing framework:\n"
    "5. Dice are optional resolution tools, not the default response to every action.\n"
    "6. Only call for a check when outcome is uncertain and failure has meaningful consequences.\n"
    "7. Forced checks: use when danger is involuntary or immediate (poison, trap trigger, collapse).\n"
    "8. Scene checks: use sparingly on area entry; avoid repeated unsolicited scene checks.\n"
    "9. Intent checks: for interactive obstacles, wait for player approach before rolling.\n"
    "10. If approach is missing, call prompt_player_action first with an open DM question.\n"
    "11. After intent is declared, choose skill/DC and call roll_dice with check_type='intent_check' and player_intent set.\n"
    "12. In roll_dice, provide dm_prompt that sounds like a DM calling the check in-fiction.\n\n"
    "Mechanics discipline:\n"
    "13. Never fabricate roll outcomes. Use tools for mechanical resolution.\n"
    "14. If no check is needed, resolve narratively without rolling.\n"
    "15. When a check is needed, use: get_skill_stat -> get_stat_modifier -> roll_dice -> record_skill_check.\n"
    "16. Do not enforce any check quota. End naturally when the scene reaches a good pause."
)


## Step 7 - Main Interactive Loop

Runs the model/tool loop, feeds tool outputs back into history, and enforces completion/style gates.


In [7]:
# Section: Main loop
# Purpose: Orchestrate iterative DM turns with policy checks and player-input tools.

def run_agency_dm_loop(user_prompt: str, max_iterations: int = MAX_ITERATIONS) -> dict[str, Any]:
    reset_run_state()

    trace_lines: list[str] = []
    response_snapshots: list[dict[str, Any]] = []
    start = time.time()

    messages: list[dict[str, Any]] = [
        {"role": "user", "content": "Character sheet (source of truth):\n" + json.dumps(CHARACTER_SHEET, indent=2)},
        {"role": "user", "content": "World state at scene start:\n" + world_summary()},
        {"role": "user", "content": user_prompt},
    ]

    final_answer = ""

    trace_print("RUN", f"model={MODEL} max_iterations={max_iterations}", trace_lines)

    for iteration in range(1, max_iterations + 1):
        trace_print("ITER", f"{iteration}/{max_iterations}", trace_lines)

        try:
            response = adapter.request_with_tools(
                stage="notebook_05_player_input_agency",
                system_prompt=SYSTEM_PROMPT,
                messages=messages,
                tools=TOOLS,
            )
        except LLMError as exc:
            trace_print("MODEL-ERROR", str(exc), trace_lines)
            trace_print("MODEL-ERROR", f"Configured model is '{MODEL}'. Check `ollama list` and MODEL.", trace_lines)
            return {
                "status": "error",
                "error": str(exc),
                "messages": messages,
                "trace_lines": trace_lines,
                "response_snapshots": response_snapshots,
                "progress": get_progress(),
                "check_log": list(RUN_STATE["check_log"]),
                "duration_s": round(time.time() - start, 3),
            }

        response_dict = response_to_dict(response)
        response_snapshots.append(response_dict)

        if SHOW_RESPONSE_STATS:
            stats = {
                "model": response_dict.get("model"),
                "done_reason": response_dict.get("done_reason"),
                "prompt_eval_count": response_dict.get("prompt_eval_count"),
                "eval_count": response_dict.get("eval_count"),
                "total_duration": response_dict.get("total_duration"),
            }
            trace_print("MODEL-STATS", json.dumps(stats, ensure_ascii=True), trace_lines)

        if SHOW_RAW_RESPONSE:
            trace_print("RAW-RESPONSE", json.dumps(response_dict, indent=2, ensure_ascii=True), trace_lines)

        assistant_text = adapter._extract_content(response).strip()
        thinking_text = extract_thinking_text(response).strip()

        trace_print("ASSISTANT-CONTENT", preview(assistant_text or "<empty>"), trace_lines)
        if SHOW_THINKING_TRACE:
            trace_print("ASSISTANT-THINKING", preview(thinking_text or "<none>"), trace_lines)

        tool_calls = normalize_tool_calls(response)

        if not tool_calls:
            fallback_source = "\n".join(x for x in [assistant_text, thinking_text] if x)
            fallback_calls = extract_fallback_tool_calls_from_text(fallback_source)
            if fallback_calls:
                tool_calls = fallback_calls
                trace_print("FALLBACK", f"Recovered {len(tool_calls)} text-encoded tool calls.", trace_lines)

        if tool_calls:
            messages.append(
                {
                    "role": "assistant",
                    "content": assistant_text,
                    "tool_calls": [c["raw"] for c in tool_calls],
                }
            )

            for call in tool_calls:
                trace_print("TOOL-SOURCE", f"{call['name']} via {call.get('source', 'unknown')}", trace_lines)
                payload = execute_tool(call["name"], call["arguments"], trace_lines)
                messages.append(
                    {
                        "role": "tool",
                        "tool_name": call["name"],
                        "content": json.dumps(payload, separators=(",", ":"), ensure_ascii=True),
                    }
                )

                if payload.get("hook_note"):
                    messages.append(
                        {
                            "role": "user",
                            "content": "Runtime policy note: " + str(payload["hook_note"]),
                        }
                    )

            progress = get_progress()
            trace_print(
                "PROGRESS",
                " ".join(
                    [
                        f"tool_calls={progress['total_tool_calls']}",
                        f"completed_checks={progress['completed_checks']}",
                        f"rolls_requested={progress['rolls_requested']}",
                        f"actions_prompted={progress['actions_prompted']}",
                    ]
                ),
                trace_lines,
            )
            continue

        progress = get_progress()

        policy_reason = stop_policy(assistant_text)
        if policy_reason:
            reason_text = policy_reason
            trace_print("STOP-BLOCK", reason_text, trace_lines)
            messages.append({"role": "assistant", "content": assistant_text})
            messages.append(
                {
                    "role": "user",
                    "content": "Completion blocked by runtime policy. " + reason_text + " Continue and comply.",
                }
            )
            continue

        messages.append({"role": "assistant", "content": assistant_text})
        final_answer = assistant_text
        return {
            "status": "completed",
            "final_answer": final_answer,
            "rounds": iteration,
            "messages": messages,
            "trace_lines": trace_lines,
            "response_snapshots": response_snapshots,
            "progress": progress,
            "check_log": list(RUN_STATE["check_log"]),
            "duration_s": round(time.time() - start, 3),
        }

    return {
        "status": "max_iterations",
        "final_answer": final_answer or "Stopped due to max iterations.",
        "messages": messages,
        "trace_lines": trace_lines,
        "response_snapshots": response_snapshots,
        "progress": get_progress(),
        "check_log": list(RUN_STATE["check_log"]),
        "duration_s": round(time.time() - start, 3),
    }


## Step 8 - Demo Run

Run this cell. You may be prompted for player actions and/or d20 rolls depending on how the model drives the scene.


In [8]:
# Section: Demo run
# Purpose: Execute the full loop and inspect check logs + trace output.

result = run_agency_dm_loop(ENCOUNTER_PROMPT)

print("\n===== RUN RESULT =====")
print("status:", result.get("status"))
print("rounds:", result.get("rounds"))
print("duration_s:", result.get("duration_s"))
print("progress:", json.dumps(result.get("progress", {}), indent=2))

print("\n===== FINAL ANSWER =====")
print(result.get("final_answer", ""))

print("\n===== CHECK LOG =====")
for entry in result.get("check_log", []):
    print(json.dumps(entry, ensure_ascii=True))

trace_lines = result.get("trace_lines", [])
print("\nTRACE LINE COUNT:", len(trace_lines))
if PRINT_FULL_TRACE_AFTER_RUN:
    for line in trace_lines:
        print(line)

snapshots = result.get("response_snapshots", [])
print("\nRESPONSE SNAPSHOT COUNT:", len(snapshots))
if PRINT_FULL_RESPONSE_SNAPSHOTS_AFTER_RUN:
    for i, snap in enumerate(snapshots, start=1):
        print(f"\n--- SNAPSHOT {i} ---")
        print(json.dumps(snap, indent=2, ensure_ascii=True))


[12:54:58] [RUN] model=gpt-oss:20b max_iterations=80
[12:54:58] [ITER] 1/80
[12:55:10] [MODEL-STATS] {"model": "gpt-oss:20b", "done_reason": "stop", "prompt_eval_count": 1115, "eval_count": 521, "total_duration": 11038837083}
[12:55:10] [RAW-RESPONSE] {
  "model": "gpt-oss:20b",
  "created_at": "2026-02-20T17:55:10.012146Z",
  "done": true,
  "done_reason": "stop",
  "total_duration": 11038837083,
  "load_duration": 108191541,
  "prompt_eval_count": 1115,
  "prompt_eval_duration": 1044948375,
  "eval_count": 521,
  "eval_duration": 9780941374,
  "message": {
    "role": "assistant",
    "content": "",
    "thinking": "We need to start the scene. The instructions: \"Begin the Shattered Beacon scene with improvisational pacing. Keep dice optional and only call for rolls when uncertainty and stakes justify it. Do not steer me toward any single obstacle; follow my declared intent and let me ignore or revisit hooks freely.\" So we need to narrate the setting, mention elements, and prompt th